### Задание 1
Напишите функцию, которая классифицирует фильмы из материалов занятия по правилам:

оценка 2 и ниже — низкий рейтинг;
оценка 4 и ниже — средний рейтинг;
оценка 4.5 и 5 — высокий рейтинг.
Результат классификации запишите в столбец class.

In [13]:
import pandas as pd
movies = pd.read_csv ('movies.csv')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [14]:
ratings = pd.read_csv ('ratings.csv')
ratings.head()


,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [29]:
def classify (param):
    """ Определение рейтинга по оценке """
    if param <= 2:
        return 'низкий рейтинг'
    elif param <=4:
        return 'средний рейтинг'
    elif param <= 5:
        return 'высокий рейтинг'
    else:
        return 'рейтинг не определен'

ratings ['class'] = ratings['rating'].apply(classify)
ratings.head (10)

,userId,movieId,rating,timestamp,class
0,1,31,2.5,1260759144,средний рейтинг
1,1,1029,3.0,1260759179,средний рейтинг
2,1,1061,3.0,1260759182,средний рейтинг
3,1,1129,2.0,1260759185,низкий рейтинг
4,1,1172,4.0,1260759205,средний рейтинг
5,1,1263,2.0,1260759151,низкий рейтинг
6,1,1287,2.0,1260759187,низкий рейтинг
7,1,1293,2.0,1260759148,низкий рейтинг
8,1,1339,3.5,1260759125,средний рейтинг
9,1,1343,2.0,1260759131,низкий рейтинг


In [28]:

movies['ave_rating'] = ratings.groupby('movieId')['rating'].mean()
movies['class'] = movies['ave_rating'].apply(classify)
movies.head()


,movieId,title,genres,ave_rating,class
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,NaN,высокий рейтинг
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.872470,средний рейтинг
2,3,Grumpier Old Men (1995),Comedy|Romance,3.401869,средний рейтинг
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,3.161017,средний рейтинг
4,5,Father of the Bride Part II (1995),Comedy,2.384615,средний рейтинг


In [25]:
# проверяем откуда взялся NaNcheck = ratings [ratings['movieId'] == 1] 

print (check)

       userId  movieId  rating   timestamp            class
495         7        1     3.0   851866703  средний рейтинг
699         9        1     4.0   938629179  средний рейтинг
889        13        1     5.0  1331380058  высокий рейтинг
962        15        1     2.0   997938310   низкий рейтинг
3105       19        1     3.0   855190091  средний рейтинг
...       ...      ...     ...         ...              ...
98531     660        1     2.5  1436680062  средний рейтинг
98714     663        1     4.0  1438397999  средний рейтинг
98740     664        1     3.5  1362421730  средний рейтинг
99858     670        1     4.0   938782344  средний рейтинг
99889     671        1     5.0  1064891129  высокий рейтинг

[247 rows x 5 columns]


In [26]:
#избавляемся от NaN с помощью заполнения средним значением
mean_ratings = ratings.groupby('movieId')['rating'].mean()
movies['ave_rating'] = movies['movieId'].map(mean_ratings).fillna(mean_rating)
movies['class'] = movies['ave_rating'].apply(classify)
movies.head()

,movieId,title,genres,ave_rating,class
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.872470,средний рейтинг
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.401869,средний рейтинг
2,3,Grumpier Old Men (1995),Comedy|Romance,3.161017,средний рейтинг
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.384615,средний рейтинг
4,5,Father of the Bride Part II (1995),Comedy,3.267857,средний рейтинг
5,6,Heat (1995),Action|Crime|Thriller,3.884615,средний рейтинг
6,7,Sabrina (1995),Comedy|Romance,3.283019,средний рейтинг
7,8,Tom and Huck (1995),Adventure|Children,3.800000,средний рейтинг
8,9,Sudden Death (1995),Action,3.150000,средний рейтинг
9,10,GoldenEye (1995),Action|Adventure|Thriller,3.450820,средний рейтинг


### Задание 2
Используйте файл keywords.csv.
Нужно написать гео-классификатор, который каждой строке сможет выставить географическую при надлежность определённому региону. Т. е. если поисковый запрос содержит название города региона, то в столбце ‘region’ пишется название этого региона. Если поисковый запрос не содержит названия города, то ставим ‘undefined’.

Правила распределения по регионам Центр, Северо-Запад и Дальний Восток:


In [36]:
geo_data = {
'Центр': ['москва', 'тула', 'ярославль'],
'Северо-Запад': ['петербург', 'псков', 'мурманск'],
'Дальний Восток': ['владивосток', 'сахалин', 'хабаровск']
}


Результат классификации запишите в отдельный столбец region.

In [40]:
keywords = pd.read_csv ('keywords.csv')
keywords.head()

,keyword,shows
0,вк,64292779
1,одноклассники,63810309
2,порно,41747114
3,ютуб,39995567
4,вконтакте,21014195


In [41]:
def classify_region(text):
    text = text.lower()
    for region, cities in geo_data.items():
        for city in cities:
            if city in text:
                return region
    return 'undefined'
    
keywords['region'] = keywords['keyword'].apply(classify_region)
keywords.head ()


,keyword,shows,region
0,вк,64292779,undefined
1,одноклассники,63810309,undefined
2,порно,41747114,undefined
3,ютуб,39995567,undefined
4,вконтакте,21014195,undefined
